In [1]:
%matplotlib inline
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import re
from tqdm import tqdm
import traceback

params = {
    'totP': {
        'ranges': {
            'GM': [0.7, 0.9],
            'WM': [0.875, 0.975]
        },
        'ticks_nb': 5
    },
    'linR': {
        'ranges': {
            'GM': [0, 20],
            'WM': [0, 40]
        },
        'ticks_nb': 11
    },
    'azimuth_local_var': {
        'ranges': {
            'GM': [0, 60],
            'WM': [0, 60]
        },
        'ticks_nb': 6
    }
}
times = ['T0', 'T1', 'T2']

temperature_folder_path = Path("/media/elea/2TBBackUp/Arnas/Temperature_v2")
pattern = re.compile(r'^(GM|WM)_\d+.*$')  # allows GM/WM + number + optional suffix

for measurement in tqdm(temperature_folder_path.iterdir(), total = len(list(temperature_folder_path.iterdir()))):
    if 'T0' in measurement.name:
        _50x50_folder = measurement / "polarimetry" / "550nm" / "50x50_images"
        mm_file = measurement / "polarimetry" / "550nm" / "MM.npz"
        if not mm_file.exists():
            continue
        MMs = {}
        MMs[measurement.name] = np.load(mm_file)

        for time in times:
            mm_file = measurement.parents[0] / measurement.name.replace(times[0], time) / "polarimetry" / "550nm" / "MM.npz"
            MMs[measurement.name.replace(times[0], time)] = np.load(mm_file)

        for param, _ in tqdm(params.items(), total = len(params)):
            
            for ROI in _50x50_folder.iterdir():
                
                if ROI.is_dir() and pattern.match(ROI.name):

                    for time in times:
                        output_folder = _50x50_folder / (measurement.name.replace(times[0], time) + "_distributions")
                        output_folder.mkdir(exist_ok=True)

                        totP = MMs[measurement.name.replace(times[0], time)][param]
                        
                        mask_path = ROI / (measurement.name.replace(times[0], time) + "_selected.png")
                        if not mask_path.exists():
                            continue
                        
                        mask = cv2.imread(str(mask_path), 0) == 0
                        masked_values = totP[mask]
                        
                        if masked_values.size == 0:
                            continue  # skip empty masks
                        
                        if "GM" in ROI.name:
                            x_min, x_max = params[param]["ranges"]["GM"]
                        else:  # assume WM
                            x_min, x_max = params[param]["ranges"]["WM"]
    
                        try:
                            masked_values_clipped = masked_values[(masked_values >= x_min) & (masked_values <= x_max)]
                            if masked_values_clipped.size == 0:
                                continue
                            
                            # KDE
                            kde = gaussian_kde(masked_values_clipped)
                            x = np.linspace(x_min, x_max, 1000)
                            y = kde(x)

                            mean_val = masked_values_clipped.mean()
                            median_val = np.median(masked_values_clipped)
                            x_max_y = x[np.argmax(y)]  # x at which KDE is maximum

                            # Plot line only
                            plt.figure(figsize=(6,4))
                            text_str = f"mean: {mean_val:.2f}\nmedian: {median_val:.2f}\nmax: {x_max_y:.2f}"
                            plt.text(
                                0.82, 0.95, text_str, transform=plt.gca().transAxes, 
                                fontsize=14, verticalalignment='top', horizontalalignment='center',
                                fontweight='bold'
                            )
                            plt.plot(x, y / max(y), label=f"{ROI.name}")
                            plt.title(f"KDE of {param} in {ROI.name}")
                            plt.xlim(x_min, x_max)
                            xticks = np.linspace(x_min, x_max, params[param]["ticks_nb"])
                            plt.xticks(xticks, fontweight='bold', fontsize = 12)
                            plt.yticks(fontweight='bold', fontsize = 12)

                            plt.savefig(output_folder / (param + "_" + ROI.name + ".png"))
                            plt.close()
                        except:
                            traceback.print_exc()

  0%|                                                     | 0/3 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/tmp/ipykernel_9220/4206091712.py", line 85, in <module>
    kde = gaussian_kde(masked_values_clipped)
  File "/home/elea/Documents/HORAO/venvs/riap/lib/python3.13/site-packages/scipy/stats/_kde.py", line 211, in __init__
    raise ValueError("`dataset` input should have multiple elements.")
ValueError: `dataset` input should have multiple elements.

 33%|███████████████                              | 1/3 [00:25<00:51, 25.65s/it]Traceback (most recent call last):
  File "/tmp/ipykernel_9220/4206091712.py", line 85, in <module>
    kde = gaussian_kde(masked_values_clipped)
  File "/home/elea/Documents/HORAO/venvs/riap/lib/python3.13/site-packages/scipy/stats/_kde.py", line 211, in __init__
    raise ValueError("`dataset` input should have multiple elements.")
ValueError: `dataset` input should have multiple elements.

 67%|██████████████████████████████             